# MEP Tutorial Part 2: Graph-Based MEP Network Design

This notebook demonstrates creating MEP networks using graph-based topology,
where vertices represent connection points and edges represent distribution paths.

**Adapted from topologicpy MEP02 notebook**

This program is free software under the GNU Affero General Public License.

## What You'll Learn

1. Creating graph-based MEP networks with outpost connections
2. Visualizing distribution networks
3. Analyzing network topology

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Part 1: Define MEP Distribution Network

We create a distribution network where:
- Each node represents a junction/connection point
- Edges represent pipes/ducts
- The network has a tree-like structure from source to endpoints

```
                    F
                   /
              C---
             /
    A---B---+---D---G
             \
              E---H---I
                   \
                    J
```

In [ ]:
# Define network nodes with their positions and connections (outposts)
# Format: (node_id, x, y, z, outposts)

nodes_data = [
    ("A", 0, 0, 0, ["B"]),           # Source node
    ("B", 10, 0, 0, ["C", "D", "E"]), # Main distribution hub
    ("C", 15, 5, 0, ["F"]),           # Branch point
    ("D", 15, 0, 0, ["G"]),           # Branch point
    ("E", 15, -5, 0, ["H"]),          # Branch point
    ("F", 20, 5, 0, []),              # Terminal node
    ("G", 20, 0, 0, []),              # Terminal node
    ("H", 20, -5, 0, ["I", "J"]),     # Secondary distribution
    ("I", 25, -3, 0, []),             # Terminal node
    ("J", 25, -7, 0, []),             # Terminal node
]

# Create vertices
vertices = {}
for node_id, x, y, z, _ in nodes_data:
    vertices[node_id] = tf.Vertex.ByCoordinates(x, y, z)

print(f"Created {len(vertices)} network nodes:")
for node_id, v in vertices.items():
    coords = v.Coordinates()
    print(f"  {node_id}: ({coords[0]:.1f}, {coords[1]:.1f}, {coords[2]:.1f})")

In [ ]:
# Create edges based on outpost connections
edges = []
edge_info = []  # Store edge information for analysis

for node_id, x, y, z, outposts in nodes_data:
    source_vertex = vertices[node_id]
    for target_id in outposts:
        if target_id in vertices:
            target_vertex = vertices[target_id]
            edge = tf.Edge.ByStartVertexEndVertex(source_vertex, target_vertex)
            edges.append(edge)
            length = edge.Length()
            edge_info.append((node_id, target_id, length))

print(f"\nCreated {len(edges)} edges:")
for source, target, length in edge_info:
    print(f"  {source} -> {target}: {length:.2f} units")

## Part 2: Create the Network Graph

In [ ]:
# Create graph from vertices and edges
vertex_list = list(vertices.values())
graph = tf.Graph.ByVerticesEdges(vertex_list, edges)

print("MEP Distribution Network:")
print("=" * 40)
print(f"  Junction points: {graph.Order()}")
print(f"  Pipe/duct segments: {graph.Size()}")
print(f"  Network density: {graph.Density():.3f}")
print(f"  Network diameter: {graph.Diameter()} steps")
print(f"  Max branching: {graph.MaximumDelta()} connections")
print(f"  Min connections: {graph.MinimumDelta()}")
print(f"  Is tree (bipartite): {graph.IsBipartite()}")

## Part 3: Visualize the Network

In [ ]:
def visualize_mep_network_2d(nodes_data, edges, edge_info):
    """Create a 2D schematic view of the MEP network."""
    fig = go.Figure()
    
    # Plot edges with arrows
    for edge in edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            
            # Main line
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='blue', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
            
            # Arrow marker at 80% along edge
            ax = p1[0] + 0.8 * (p2[0] - p1[0])
            ay = p1[1] + 0.8 * (p2[1] - p1[1])
            
            fig.add_trace(go.Scatter(
                x=[ax], y=[ay],
                mode='markers',
                marker=dict(
                    symbol='triangle-right',
                    size=12,
                    color='darkblue',
                    angle=np.degrees(np.arctan2(p2[1]-p1[1], p2[0]-p1[0]))
                ),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plot nodes
    node_types = {
        'source': ('green', 20, 'Source'),
        'hub': ('orange', 15, 'Hub'),
        'branch': ('yellow', 12, 'Branch'),
        'terminal': ('red', 10, 'Terminal')
    }
    
    for node_id, x, y, z, outposts in nodes_data:
        # Determine node type
        if node_id == 'A':
            ntype = 'source'
        elif len(outposts) >= 3:
            ntype = 'hub'
        elif len(outposts) > 0:
            ntype = 'branch'
        else:
            ntype = 'terminal'
        
        color, size, label = node_types[ntype]
        
        fig.add_trace(go.Scatter(
            x=[x], y=[y],
            mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='black', width=1)),
            text=[node_id],
            textposition='top center',
            textfont=dict(size=14, color='black'),
            name=f"{node_id} ({label})",
            hoverinfo='name'
        ))
    
    fig.update_layout(
        title='MEP Distribution Network (Schematic)',
        xaxis=dict(title='X Position', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y Position'),
        width=900,
        height=500,
        showlegend=True,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig_2d = visualize_mep_network_2d(nodes_data, edges, edge_info)
fig_2d.show()

## Part 4: Network Analysis

In [ ]:
# Analyze each node
print("Node Analysis:")
print("=" * 50)

graph_vertices = graph.Vertices()

# Map coordinates to node IDs
coord_to_id = {}
for node_id, x, y, z, _ in nodes_data:
    coord_to_id[(x, y, z)] = node_id

def get_node_id(vertex):
    coords = vertex.Coordinates()
    key = (round(coords[0]), round(coords[1]), round(coords[2]))
    return coord_to_id.get(key, f"Unknown")

for v in graph_vertices:
    node_id = get_node_id(v)
    degree = graph.VertexDegree(v)
    adjacent = graph.AdjacentVertices(v)
    adj_ids = [get_node_id(av) for av in adjacent]
    
    print(f"\n{node_id}:")
    print(f"  Degree: {degree}")
    print(f"  Connected to: {', '.join(adj_ids)}")

In [ ]:
# Find paths from source to all terminal nodes
source_vertex = vertices['A']
terminal_nodes = ['F', 'G', 'I', 'J']

print("\nPaths from Source (A) to Terminal Nodes:")
print("=" * 50)

for terminal_id in terminal_nodes:
    terminal_vertex = vertices[terminal_id]
    distance = graph.Distance(source_vertex, terminal_vertex)
    path = graph.Path(source_vertex, terminal_vertex)
    
    if path:
        path_vertices = path.Vertices()
        path_ids = [get_node_id(pv) for pv in path_vertices]
        
        # Calculate total path length
        path_edges = path.Edges()
        total_length = sum(e.Length() for e in path_edges)
        
        print(f"\nA -> {terminal_id}:")
        print(f"  Hops: {distance}")
        print(f"  Length: {total_length:.2f} units")
        print(f"  Route: {' -> '.join(path_ids)}")
    else:
        print(f"\nA -> {terminal_id}: No path found")

## Part 5: Distribution Hierarchy (Depth Map)

In [ ]:
# Create depth map from source
depth_map = graph.DepthMap(source_vertex)

print("Distribution Hierarchy (from Source A):")
print("=" * 40)

# Group nodes by depth level
levels = {}
for i, depth in enumerate(depth_map):
    node_id = get_node_id(graph_vertices[i])
    if depth not in levels:
        levels[depth] = []
    levels[depth].append(node_id)

for level in sorted(levels.keys()):
    nodes = levels[level]
    print(f"  Level {level}: {', '.join(sorted(nodes))}")

In [ ]:
def visualize_with_hierarchy(nodes_data, edges, depth_map, graph_vertices):
    """Visualize network colored by hierarchy level."""
    fig = go.Figure()
    
    # Color palette for levels
    colors = ['green', 'orange', 'yellow', 'pink', 'red']
    
    # Create depth lookup
    depth_lookup = {}
    for i, v in enumerate(graph_vertices):
        node_id = get_node_id(v)
        depth_lookup[node_id] = depth_map[i]
    
    # Plot edges
    for edge in edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='gray', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plot nodes colored by level
    for node_id, x, y, z, outposts in nodes_data:
        level = depth_lookup.get(node_id, 0)
        color = colors[min(level, len(colors)-1)]
        
        fig.add_trace(go.Scatter(
            x=[x], y=[y],
            mode='markers+text',
            marker=dict(size=25, color=color, line=dict(color='black', width=2)),
            text=[node_id],
            textposition='middle center',
            textfont=dict(size=12, color='black'),
            name=f"{node_id} (Level {level})",
            hoverinfo='name'
        ))
    
    fig.update_layout(
        title='MEP Network by Distribution Level',
        xaxis=dict(title='X Position', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y Position'),
        width=900,
        height=500,
        showlegend=True,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig_hierarchy = visualize_with_hierarchy(nodes_data, edges, depth_map, graph_vertices)
fig_hierarchy.show()

## Part 6: Pipe/Duct Sizing Concept

In real MEP design, pipe/duct sizes depend on:
- Flow rate requirements
- Pressure drop constraints
- Number of downstream connections

Here we demonstrate a simplified sizing based on connectivity.

In [ ]:
# Calculate downstream count for each edge
def count_downstream(graph, vertex, visited=None):
    """Count downstream nodes from a vertex."""
    if visited is None:
        visited = set()
    
    node_id = get_node_id(vertex)
    if node_id in visited:
        return 0
    
    visited.add(node_id)
    count = 1
    
    adjacent = graph.AdjacentVertices(vertex)
    for adj_v in adjacent:
        adj_id = get_node_id(adj_v)
        if adj_id not in visited:
            count += count_downstream(graph, adj_v, visited)
    
    return count

print("Pipe/Duct Sizing by Downstream Count:")
print("=" * 50)
print("(Higher downstream count = larger pipe/duct)")
print()

for source_id, target_id, length in edge_info:
    target_vertex = vertices[target_id]
    downstream = count_downstream(graph, target_vertex)
    
    # Simplified sizing
    if downstream >= 4:
        size = "Large (main trunk)"
    elif downstream >= 2:
        size = "Medium (branch)"
    else:
        size = "Small (terminal)"
    
    print(f"{source_id} -> {target_id}:")
    print(f"  Length: {length:.1f} units")
    print(f"  Downstream nodes: {downstream}")
    print(f"  Suggested size: {size}")
    print()

## Summary

In this tutorial, we covered:

1. **Graph-Based Network Design**: Creating MEP networks with explicit connections
2. **Network Visualization**: Schematic and hierarchical views
3. **Path Analysis**: Finding routes from source to terminals
4. **Hierarchy Mapping**: Understanding distribution levels
5. **Sizing Concepts**: Using topology for preliminary sizing

### Not Yet Implemented in topologic_fast:
- `Graph.ByTopology(..., toOutposts=True)` - Creating graphs from outpost relationships
- `Topology.SetDictionary()` - Storing attributes on elements
- `Graph.PyvisGraph()` - Exporting to interactive HTML visualization

### Applications:
- HVAC duct network design
- Plumbing distribution systems
- Fire suppression systems
- Electrical distribution networks